# 📄 IntelliCode-SL | Docs + Explanation SLM Fine-Tuning
**Model:** `meta-llama/Llama-3.2-1B-Instruct`  
**Tasks:** `explain` | `document`

> ⚠️ You must accept Meta's license first: [Llama-3.2-1B-Instruct on HuggingFace](https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct)  
> Run cells top to bottom. GPU runtime required.

In [ ]:
# ── Cell 1: Install Dependencies ──────────────────────────────
!pip install -q unsloth transformers datasets peft accelerate bitsandbytes trl
print("✅ Dependencies installed")

In [ ]:
# ── Cell 2: Imports ───────────────────────────────────────────
import os, torch
from google.colab import drive
from huggingface_hub import login
from datasets import load_dataset, concatenate_datasets
from transformers import TrainingArguments
from unsloth import FastLanguageModel
from trl import SFTTrainer
print("✅ Imports done | GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NOT FOUND")

In [ ]:
# ── Cell 3: Mount Google Drive ────────────────────────────────
drive.mount("/content/drive")

DRIVE_BASE   = "/content/drive/MyDrive/IntelliCode-SL"
ADAPTER_SAVE = f"{DRIVE_BASE}/adapters/docs_explanation_adapter"

DATASET_PATHS = {
    "explain"  : f"{DRIVE_BASE}/datasets/explanation_dataset.json",
    "document" : f"{DRIVE_BASE}/datasets/documentation_dataset.json",
}

os.makedirs(ADAPTER_SAVE, exist_ok=True)
print("✅ Drive mounted")
for k, v in DATASET_PATHS.items():
    print(f"   {k:10s} → {v}")
print(f"   adapter    → {ADAPTER_SAVE}")

In [ ]:
# ── Cell 4: HuggingFace Login ──────────────────────────────────
# ⚠️ Accept Meta license first: https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct
HF_TOKEN = "hf_XXXXXXXXXXXXXXXXXXXXXXXXXX"   # ← paste your token here
login(token=HF_TOKEN)
print("✅ Logged in to HuggingFace")

In [ ]:
# ── Cell 5: Load Model via Unsloth ─────────────────────────────
MAX_SEQ_LEN = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "meta-llama/Llama-3.2-1B-Instruct",
    max_seq_length = MAX_SEQ_LEN,
    dtype          = None,
    load_in_4bit   = True,
    token          = HF_TOKEN,
)
print("✅ Model loaded")

In [ ]:
# ── Cell 6: Attach LoRA Adapter ────────────────────────────────
model = FastLanguageModel.get_peft_model(
    model,
    r              = 16,
    target_modules = ["q_proj","k_proj","v_proj","o_proj",
                      "gate_proj","up_proj","down_proj"],
    lora_alpha     = 16,
    lora_dropout   = 0.05,
    bias           = "none",
    use_gradient_checkpointing = "unsloth",
    random_state   = 42,
)
print("✅ LoRA adapter attached")
model.print_trainable_parameters()

## 📁 Dataset Format
Each JSON file should have samples like:
```json
[
  {"input": "<code here>", "output": "<explanation or documentation>"}
]
```
- `explanation_dataset.json` → code → plain English explanation  
- `documentation_dataset.json` → code → docstrings / README / comments

In [ ]:
# ── Cell 7: Load & Format Dataset ──────────────────────────────
TASK_PROMPTS = {
    "explain"  : "Explain what the following code does in simple, clear language.",
    "document" : "Generate proper documentation (docstrings, comments, or README) for the following code.",
}

EOS = tokenizer.eos_token

PROMPT_TEMPLATE = """### Task: {task_instruction}

### Code:
{input}

### Response:
{output}"""

def format_sample(sample):
    task_instr = TASK_PROMPTS.get(sample.get("task","explain"), "Respond to the following.")
    return {"text": PROMPT_TEMPLATE.format(
        task_instruction=task_instr,
        input=sample["input"],
        output=sample["output"]
    ) + EOS}

all_splits = []
for task_name, path in DATASET_PATHS.items():
    ds = load_dataset("json", data_files=path, split="train")
    ds = ds.map(lambda x, t=task_name: {**x, "task": t})
    all_splits.append(ds)
    print(f"  Loaded {len(ds):>4d} samples for '{task_name}'")

merged_dataset = concatenate_datasets(all_splits).shuffle(seed=42)
dataset        = merged_dataset.map(format_sample)
print(f"\n✅ Total samples: {len(dataset)}")

In [ ]:
# ── Cell 8: Training Arguments ─────────────────────────────────
training_args = TrainingArguments(
    output_dir                  = "/content/docs_explanation_checkpoints",
    per_device_train_batch_size = 4,
    gradient_accumulation_steps = 4,
    num_train_epochs            = 3,
    learning_rate               = 2e-4,
    fp16                        = not torch.cuda.is_bf16_supported(),
    bf16                        = torch.cuda.is_bf16_supported(),
    logging_steps               = 20,
    save_strategy               = "epoch",
    warmup_ratio                = 0.03,
    lr_scheduler_type           = "cosine",
    report_to                   = "none",
)
print("✅ Training args set")

In [ ]:
# ── Cell 9: Train ──────────────────────────────────────────────
trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = dataset,
    dataset_text_field = "text",
    max_seq_length     = MAX_SEQ_LEN,
    args               = training_args,
)

print("🚀 Starting training...")
trainer.train()
print("✅ Training complete!")

In [ ]:
# ── Cell 10: Save Adapter to Google Drive ──────────────────────
model.save_pretrained(ADAPTER_SAVE)
tokenizer.save_pretrained(ADAPTER_SAVE)
print(f"✅ Adapter saved to Drive → {ADAPTER_SAVE}")

In [ ]:
# ── Cell 11: Quick Inference Test ──────────────────────────────
FastLanguageModel.for_inference(model)

test_prompt = PROMPT_TEMPLATE.format(
    task_instruction = TASK_PROMPTS["explain"],
    input = "def factorial(n):\n    if n == 0: return 1\n    return n * factorial(n-1)",
    output = ""
)
inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=300, temperature=0.3, do_sample=True)

result = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("Explanation:")
print(result.split("### Response:")[-1].strip())